[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/40_mlops/90_exercises.ipynb)

# Bài tập — Chương 40: MLOps (L0 → L7)

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/40_mlops

In [ ]:
import hashlib
import inspect
import os
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.datasets import make_customer_churn
from ml_course.utils import repo_path

warnings.filterwarnings("ignore", message="Using `httpx` with")
FEATURES = ["tenure_months", "contract", "internet_service", "monthly_charges", "total_charges",
            "payment_method", "support_calls", "senior", "n_products"]
NUMERIC = ["tenure_months", "monthly_charges", "total_charges", "support_calls", "n_products"]
CATEGORICAL = ["contract", "internet_service", "payment_method", "senior"]


def fit_churn_model(df):
    return make_pipeline(ColumnTransformer([("num", StandardScaler(), NUMERIC), ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)]),
                         LogisticRegression(max_iter=1000)).fit(df[FEATURES], df.churn)

## L0 — Nhớ khái niệm

**Câu 1.** (a) Trôi dữ liệu và trôi khái niệm. (b) Shadow, canary, A/B test — mỗi cách trả lời câu hỏi gì? (c) Liveness và readiness. (d) Vì sao endpoint chạy mô hình scikit-learn nên khai báo `def` thay vì `async def`? (e) Độ trễ của nhãn ảnh hưởng gì tới giám sát? (f) "Nhìn trộm" A/B test là gì?

✍️ Trả lời:

## L1 — Giải thích bằng lời

**Câu 2.** Một đội giám sát 80 đặc trưng bằng kiểm định KS hằng ngày (~100.000 dự đoán/ngày), cảnh báo khi p < 0.05. Mỗi ngày có ~30 cảnh báo nên cả đội đã tắt thông báo. Ba tháng sau, phát hiện hiệu năng mô hình đã sụp từ lâu do một chính sách giá mới — và không có cảnh báo nào khác biệt so với bình thường. Giải thích hai vấn đề và thiết kế lại hệ thống giám sát.

✍️ Trả lời:

## L2 — Tính tay

**Câu 3.** Tỉ lệ trong 4 bin: reference `[0.25, 0.25, 0.25, 0.25]`, hiện tại `[0.10, 0.20, 0.30, 0.40]`. Tính PSI $= \sum (c_i - r_i)\ln(c_i/r_i)$ → `psi_manual`. Mức độ trôi theo quy ước?

In [ ]:
### START CODE HERE ### (≈ 1 dòng)
psi_manual = None
### END CODE HERE ###
print(f"PSI = {psi_manual:.4f}")

In [ ]:
check_close(psi_manual, 0.2282, tol=1e-3, message="PSI ≈ 0.228 — thay đổi vừa (0.1–0.25), sát ngưỡng lớn")

**Câu 4.** A/B test: tỉ lệ chuyển đổi hiện tại 20%, muốn phát hiện mức 25%, α = 0.05 (hai phía), power 0.8 ($z_{0.975} = 1.96$, $z_{0.8} = 0.8416$). Tính cỡ mẫu mỗi nhóm (làm tròn lên) → `n_per_group`. Nếu muốn phát hiện 20% → 22.5% (hiệu ứng bằng một nửa), cỡ mẫu xấp xỉ gấp mấy lần → `ratio`?

In [ ]:
### START CODE HERE ### (≈ 3 dòng)
p_bar = None
n_per_group = None
ratio = None
### END CODE HERE ###
print(n_per_group, ratio)

In [ ]:
check_equal(n_per_group, 1094, "cỡ mẫu mỗi nhóm")
check_equal(ratio, 4, "hiệu ứng giảm một nửa → cỡ mẫu ×4 (tỉ lệ nghịch bình phương)")

## L3 — Code

**Bài 5.** Viết `psi(reference, current, bins=10, eps=1e-4)` (bin theo **phân vị của reference**, hai đầu mở ra vô cực, kẹp tỉ lệ tối thiểu `eps`) và `psi_categorical(reference, current, eps=1e-4)` (tính cả danh mục chỉ xuất hiện ở một phía).

In [ ]:
def psi(reference, current, bins=10, eps=1e-4):
    ### START CODE HERE ### (≈ 6 dòng)
    return 0.0
    ### END CODE HERE ###


def psi_categorical(reference, current, eps=1e-4):
    ### START CODE HERE ### (≈ 5 dòng)
    return 0.0
    ### END CODE HERE ###


r5 = np.random.default_rng(5)
base5 = r5.normal(50, 10, 20000)
print(f"cùng phân phối: {psi(base5, r5.normal(50, 10, 20000)):.4f} | lệch 0.5σ: {psi(base5, r5.normal(55, 10, 20000)):.4f} | lệch 1σ: {psi(base5, r5.normal(60, 10, 20000)):.4f}")

In [ ]:
check_less(psi(base5, r5.normal(50, 10, 20000)), 0.01, "cùng phân phối → PSI ≈ 0")
check_close(psi(base5, base5), 0.0, message="giống hệt → 0")
check_greater(psi(base5, r5.normal(60, 10, 20000)), 0.25, "lệch 1 độ lệch chuẩn → trôi lớn")
check_greater(psi(base5, base5 * 2), psi(base5, base5 * 1.1), "lệch càng nhiều PSI càng lớn")
check_true(np.isfinite(psi(base5, np.full(1000, 1e6))), "bin rỗng không gây vô cực (nhờ eps)")
check_close(psi_categorical(["a", "b"] * 50, ["a", "b"] * 50), 0.0, message="danh mục giống hệt")
check_close(psi_categorical(["a"] * 50 + ["b"] * 50, ["a"] * 25 + ["b"] * 75), 0.25 * np.log(0.5) * -1 + 0.25 * np.log(1.5), message="hai danh mục")
check_greater(psi_categorical(["a", "b"] * 50, ["a", "b", "c", "c"] * 25), 1.0, "danh mục mới xuất hiện → trôi rất lớn")

**Bài 6.** Viết `create_app(model, version)` trả về một ứng dụng FastAPI phục vụ mô hình phân loại hoa Iris: `GET /health` → `{"status": "ok", "model_version": version}`; `POST /predict` nhận **danh sách** hoa (4 trường `sepal_length`, `sepal_width`, `petal_length`, `petal_width` — số thực > 0 và < 30, cấm trường lạ), trả danh sách `{"species": ..., "probability": ..., "model_version": ...}` (xác suất của lớp dự đoán).

In [ ]:
try:
    from fastapi import FastAPI
    from fastapi.testclient import TestClient
    from pydantic import BaseModel, ConfigDict, Field
except ImportError as e:
    raise ImportError('Bài này cần FastAPI: pip install -e ".[mlops]"') from e

iris = pd.read_csv(repo_path("data", "sample", "iris.csv"))
IRIS_FEATURES = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
iris_model = LogisticRegression(max_iter=1000).fit(iris[IRIS_FEATURES], iris.species)


def create_app(model, version: str):
    ### START CODE HERE ### (≈ 18 dòng)
    app = FastAPI()
    return app
    ### END CODE HERE ###


client6 = TestClient(create_app(iris_model, "2.0.1"))
sample6 = iris[IRIS_FEATURES].iloc[[0, 60, 120]].to_dict("records")
print(client6.get("/health").json())
print(client6.post("/predict", json=sample6).json())

In [ ]:
check_equal(client6.get("/health").json(), {"status": "ok", "model_version": "2.0.1"}, "health")
resp6 = client6.post("/predict", json=iris[IRIS_FEATURES].to_dict("records")).json()
check_equal([r["species"] for r in resp6], list(iris_model.predict(iris[IRIS_FEATURES])), "parity: nhãn khớp mô hình")
check_close([r["probability"] for r in resp6], iris_model.predict_proba(iris[IRIS_FEATURES]).max(axis=1), message="parity: xác suất khớp")
check_equal({r["model_version"] for r in resp6}, {"2.0.1"}, "mọi dự đoán mang phiên bản")
check_equal(client6.post("/predict", json=[{**sample6[0], "sepal_length": -1}]).status_code, 422, "giá trị âm → 422")
check_equal(client6.post("/predict", json=[{**sample6[0], "color": "blue"}]).status_code, 422, "trường lạ → 422")
check_equal(client6.post("/predict", json=[{k: v for k, v in sample6[0].items() if k != "petal_width"}]).status_code, 422, "thiếu trường → 422")

## L4 — Debug

**Bài 7.** Router của một A/B test dùng hàm dưới đây. Đội phân tích phát hiện cùng một người dùng lúc thấy bản v1, lúc thấy v2 — tuỳ request rơi vào server nào (mỗi server là một tiến trình Python). Tìm nguyên nhân và viết lại `assign(user_id, share=0.1, salt="exp-v2")` cho kết quả **giống nhau ở mọi tiến trình**, đúng tỉ lệ `share`, và độc lập giữa các thí nghiệm (salt).

In [ ]:
def assign_buggy(user_id, share=0.1):
    return "v2" if hash(user_id) % 1000 < share * 1000 else "v1"


def assign(user_id, share=0.1, salt="exp-v2"):
    ### START CODE HERE ### (≈ 2 dòng)
    return "v2" if hash(user_id) % 1000 < share * 1000 else "v1"
    ### END CODE HERE ###


def assignments_in_new_process(func, hash_seed, n_users=300):
    """Chạy func trong một tiến trình Python mới với PYTHONHASHSEED cho trước (mô phỏng một server khác)."""
    code = "import hashlib\n" + inspect.getsource(func) + f"\nprint(''.join('1' if {func.__name__}(f'U{{i}}') == 'v2' else '0' for i in range({n_users})))"
    out = subprocess.run([sys.executable, "-c", code], env={**os.environ, "PYTHONHASHSEED": str(hash_seed)}, capture_output=True, text=True)
    return out.stdout.strip()


buggy = [assignments_in_new_process(assign_buggy, s) for s in (1, 2)]
fixed = [assignments_in_new_process(assign, s) for s in (1, 2)]
print(f"bản lỗi: {sum(a != b for a, b in zip(*buggy))}/300 người dùng đổi phiên bản giữa hai server")
print(f"bản sửa: {sum(a != b for a, b in zip(*fixed))}/300 người dùng đổi phiên bản giữa hai server")

In [ ]:
check_equal(fixed[0], fixed[1], "cùng kết quả ở mọi tiến trình")
check_true(buggy[0] != buggy[1], "tái hiện lỗi: hash() đổi theo PYTHONHASHSEED")
share7 = np.mean([assign(f"U{i}") == "v2" for i in range(50000)])
check_less(abs(share7 - 0.1), 0.006, "tỉ lệ v2 ≈ 10%")
overlap7 = np.mean([assign(f"U{i}", salt="other") == "v2" for i in range(50000) if assign(f"U{i}") == "v2"])
check_less(abs(overlap7 - 0.1), 0.02, "salt khác → gán độc lập")

## L5 — Phân tích

**Bài 8.** **Chọn cửa sổ huấn luyện lại.** Dòng dữ liệu 12 tuần (1.500 khách/tuần); từ tuần 7, quan hệ đổi (hệ số `support_calls` từ +0.38 thành −0.3 — chính sách tặng ưu đãi cho người gọi tổng đài). Đánh giá trên **tuần 12** các mô hình huấn luyện trên $k$ tuần gần nhất (tuần $12 - k$ … 11) với $k \in \{1, 2, 4, 8, 11\}$, và mô hình cũ (huấn luyện trên tuần 1–6). Lưu `window_results` (DataFrame index $k$ hoặc `"cũ"`, cột `AUC`, `Brier`). Nhận xét.

In [ ]:
def churn_logit(df, calls_effect):
    return (-1.6 + 1.3 * (df.contract == "month-to-month") - 0.5 * (df.contract == "two_year") - 0.035 * df.tenure_months
            + 0.018 * (df.monthly_charges - 70) + calls_effect * df.support_calls + 0.45 * (df.internet_service == "fiber")
            + 0.35 * (df.payment_method == "e-wallet") + 0.3 * df.senior - 0.2 * (df.n_products - 1))


def stream_week(week, n=1500):
    df = make_customer_churn(n, seed=300 + week).drop(columns="churn")
    rng = np.random.default_rng(700 + week)
    df["churn"] = (rng.random(n) < 1 / (1 + np.exp(-churn_logit(df, 0.38 if week < 7 else -0.3)))).astype(int)
    return df


stream = {w: stream_week(w) for w in range(1, 13)}
test8 = stream[12]
### START CODE HERE ### (≈ 7 dòng)
window_results = None
### END CODE HERE ###
print(window_results.round(4))

In [ ]:
check_equal(list(window_results.index), [1, 2, 4, 8, 11, "cũ"], "đủ các cửa sổ")
check_less(window_results.loc[4, "Brier"], window_results.loc[11, "Brier"], "cửa sổ trong giai đoạn mới tốt hơn toàn bộ lịch sử")
check_greater(window_results.loc[4, "AUC"], window_results.loc["cũ", "AUC"] + 0.03, "huấn luyện lại sau trôi khái niệm cải thiện rõ")
check_less(window_results.loc[4, "Brier"], window_results.loc[8, "Brier"], "trộn dữ liệu trước thay đổi làm kém đi")

## L6 — Thực tế

**Bài 9.** Viết `ab_test_report(conv_a, n_a, conv_b, n_b, planned_n, alpha=0.05)` → dict: `lift` ($\hat p_B - \hat p_A$), `ci_low`, `ci_high` (KTC $1-\alpha$ dùng sai số chuẩn **không gộp** $\sqrt{\hat p_A(1-\hat p_A)/n_A + \hat p_B(1-\hat p_B)/n_B}$), `p_value` (z-test hai phía, tỉ lệ **gộp**), `decision`:
- `"chưa đủ mẫu"` nếu $\min(n_A, n_B)$ < `planned_n` (chống nhìn trộm — chưa được kết luận);
- `"B tốt hơn"` / `"A tốt hơn"` nếu p < α (theo dấu của lift);
- `"không có khác biệt có ý nghĩa"` nếu ngược lại.

In [ ]:
def ab_test_report(conv_a, n_a, conv_b, n_b, planned_n, alpha=0.05):
    ### START CODE HERE ### (≈ 14 dòng)
    return {}
    ### END CODE HERE ###


for args in [(1500, 15000, 1680, 15000), (150, 1500, 190, 1500), (1500, 15000, 1530, 15000), (1700, 15000, 1500, 15000)]:
    rep = ab_test_report(*args, planned_n=14751)
    print(args, "→", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in rep.items()})

In [ ]:
rep9 = ab_test_report(1500, 15000, 1680, 15000, planned_n=14751)
check_close(rep9["lift"], 0.012, message="lift")
se9 = np.sqrt(0.1 * 0.9 / 15000 + 0.112 * 0.888 / 15000)
check_close([rep9["ci_low"], rep9["ci_high"]], [0.012 - 1.959964 * se9, 0.012 + 1.959964 * se9], tol=1e-5, message="KTC 95%")
check_close(rep9["p_value"], 2 * stats.norm.sf(0.012 / np.sqrt(0.106 * 0.894 * 2 / 15000)), message="p-value z-test gộp")
check_equal(rep9["decision"], "B tốt hơn", "đủ mẫu, có ý nghĩa")
check_equal(ab_test_report(150, 1500, 190, 1500, planned_n=14751)["decision"], "chưa đủ mẫu", "chưa đủ mẫu dù p nhỏ → không kết luận")
check_equal(ab_test_report(1500, 15000, 1530, 15000, planned_n=14751)["decision"], "không có khác biệt có ý nghĩa", "khác biệt nhỏ")
check_equal(ab_test_report(1700, 15000, 1500, 15000, planned_n=14751)["decision"], "A tốt hơn", "bản mới tệ hơn")

## L7 — Nâng cao

**Bài 10.** **Trôi trong phân phối đồng thời.** Viết `domain_auc(reference, current, seed=0)`: AUC (5-fold `cross_val_predict`, `StratifiedKFold(shuffle=True)`) của `HistGradientBoostingClassifier` phân biệt hai DataFrame số; và `univariate_auc(reference, current)` → Series AUC của bộ phân loại dùng **từng cột riêng** (cùng cách). Thử trên ba kịch bản 2 biến $(x_1, x_2)$ chuẩn hoá: (A) không trôi; (B) $x_1$ dịch 0.5; (C) **tương quan đổi dấu** (+0.8 → −0.8, phân phối biên giữ nguyên). Lưu `results10` (DataFrame index A/B/C, cột `domain`, `x1`, `x2`, `psi_max`).

In [ ]:
def scenario(kind, n=3000, seed=0):
    rng = np.random.default_rng(seed)
    corr = -0.8 if kind == "C" else 0.8
    x = rng.multivariate_normal([0, 0], [[1, corr], [corr, 1]], n)
    if kind == "B":
        x[:, 0] += 0.5
    return pd.DataFrame(x, columns=["x1", "x2"])


reference10 = scenario("ref", seed=1)


def domain_auc(reference, current, seed=0):
    ### START CODE HERE ### (≈ 5 dòng)
    return 0.5
    ### END CODE HERE ###


def univariate_auc(reference, current):
    ### START CODE HERE ### (≈ 1 dòng)
    return pd.Series(dtype=float)
    ### END CODE HERE ###


### START CODE HERE ### (≈ 5 dòng)
results10 = None
### END CODE HERE ###
print(results10.round(3))

In [ ]:
check_equal(list(results10.columns), ["domain", "x1", "x2", "psi_max"], "các cột")
check_less(results10.loc["A", "domain"], 0.56, "A: không trôi → AUC ≈ 0.5")
check_greater(results10.loc["B", "x1"], 0.6, "B: cột x1 tự nó phân biệt được")
check_greater(results10.loc["B", "x1"], results10.loc["B", "x2"] + 0.05, "B: chỉ ra đúng cột trôi")
check_less(results10.loc["C", "psi_max"], 0.05, "C: PSI từng biến không thấy gì")
check_less(results10.loc["C", ["x1", "x2"]].max(), 0.56, "C: phân loại từng biến cũng không thấy")
check_greater(results10.loc["C", "domain"], 0.8, "C: bộ phân loại miền nhiều biến phát hiện trôi")